<a href="https://colab.research.google.com/github/letruc271193-dot/projectckai/blob/main/nhandienmonan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input

train_dir = "/content/drive/MyDrive/Dataset_CNN"

img_width, img_height = 128, 128
batch_size = 32

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    brightness_range=[0.8, 1.2],
    horizontal_flip=True,
    fill_mode="nearest",
    validation_split=0.2
)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode="categorical",
    subset='training'
)

val_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode="categorical",
    subset='validation'
)

num_classes = len(train_generator.class_indices)
print(f"Phát hiện {num_classes} loại món ăn.")

base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(img_width, img_height, 3))

base_model.trainable = True


for layer in base_model.layers[:-20]:
    layer.trainable = False


model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])


model.compile(optimizer=Adam(learning_rate=0.0001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.summary()


early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)


history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=100,
    callbacks=[early_stopping, reduce_lr]
)



os.makedirs('/content/drive/MyDrive/model/', exist_ok=True)


model.save('/content/drive/MyDrive/model/model_efficientnet_pro.h5')

with open('/content/drive/MyDrive/model/labels_cnn.pkl', 'wb') as f:
    pickle.dump(train_generator.class_indices, f)

print("Đã huấn luyện xong siêu mô hình")

Found 4991 images belonging to 11 classes.
Found 1241 images belonging to 11 classes.
Phát hiện 11 loại món ăn.
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)     │ (None, 4, 4, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 11)             │         2,827 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,380,334 (16.71 MB)

 Trainable params: 1,681,723 (6.42 MB)

 Non-trainable params: 2,698,611 (10.29 MB)

Epoch 1/100
156/156 ━━━━━━━━━━━━━━━━━━━━ 1785s 11s/step - accuracy: 0.4835 - loss: 1.5929 - val_accuracy: 0.7953 - val_loss: 0.7013 - learning_rate: 1.0000e-04
Epoch 2/100
156/156 ━━━━━━━━━━━━━━━━━━━━ 99s 635ms/step - accuracy: 0.7646 - loss: 0.7107 - val_accuracy: 0.8590 - val_loss: 0.4233 - learning_rate: 1.0000e-04
Epoch 3/100
156/156 ━━━━━━━━━━━━━━━━━━━━ 98s 629ms/step - accuracy: 0.8301 - loss: 0.5134 - val_accuracy: 0.8687 - val_loss: 0.3626 - learning_rate: 1.0000e-04
Epoch 4/100
156/156 ━━━━━━━━━━━━━━━━━━━━ 97s 620ms/step - accuracy: 0.8575 - loss: 0.4277 - val_accuracy: 0.8961 - val_loss: 0.3163 - learning_rate: 1.0000e-04
Epoch 5/100
156/156 ━━━━━━━━━━━━━━━━━━━━ 96s 617ms/step - accuracy: 0.8704 - loss: 0.3735 - val_accuracy: 0.9025 - val_loss: 0.2836 - learning_rate: 1.0000e-04
Epoch 6/100
156/156 ━━━━━━━━━━━━━━━━━━━━ 96s 616ms/step - accuracy: 0.8834 - loss: 0.3457 - val_accuracy: 0.9130 - val_loss: 0.2695 - learning_rate: 1.0000e-04
Epoch 7/100
156/156 ━━━━━━━━━━━━━━━━━━━━

Đã huấn luyện xong siêu mô hình
